In [19]:
# Cell 1 — Setup

import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import shap

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "generator").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "generator").exists():
    raise ModuleNotFoundError(
        f"Could not find RingWatch generator package from {Path.cwd()}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generator.features_config import (
    PREDICTION_CUTOFF,
    FEATURES_GRAPH_PATH,
    PREDICTIONS_TEST_PATH,
    MODEL_A_PATH,
    MODEL_METRICS_PATH,
    EXPLAINABILITY_DIR,
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
    PATHS,
)

T = pd.to_datetime(PREDICTION_CUTOFF)

print("Project root:", PROJECT_ROOT)
print("Prediction cutoff:", T)
print("Model A:", MODEL_A_PATH)
print("Test predictions:", PREDICTIONS_TEST_PATH)
print("Features graph:", FEATURES_GRAPH_PATH)
print("Explainability dir:", EXPLAINABILITY_DIR)

Project root: d:\CODIN PLAYGROUND\ML-AI\RingWatch
Prediction cutoff: 2026-02-20 00:00:00
Model A: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\model\model_lgbm_A.pkl
Test predictions: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\model\model_predictions_test.csv
Features graph: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\features_graph.csv
Explainability dir: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability


In [20]:
# Cell 2 — Verify all critical files exist

critical_files = [
    MODEL_A_PATH,
    PREDICTIONS_TEST_PATH,
    FEATURES_GRAPH_PATH,
    MODEL_METRICS_PATH,
    PATHS["orders"],
    PATHS["disputes"],
    Path(PATHS["orders"]).parent / "processed" / "account_graph_edges.csv",
]

missing = [p for p in critical_files if not Path(p).exists()]

if missing:
    raise FileNotFoundError(f"Missing required 30K files:\n{missing}")

print("All required 30K files exist.")

All required 30K files exist.


In [21]:
# Cell 3 — Create explainability output directory


EXPLAINABILITY_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory ready:")
print(EXPLAINABILITY_DIR)

Output directory ready:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability


In [22]:
# Cell 4 — Load 30K Model A

with open(MODEL_A_PATH, "rb") as f:
    model_A = pickle.load(f)

model_A_features = list(model_A.feature_name_)

print("Model A loaded.")
print("Type:", type(model_A))
print("Feature count:", len(model_A_features))

Model A loaded.
Type: <class 'lightgbm.sklearn.LGBMClassifier'>
Feature count: 40


In [23]:
# Cell 5 — Load 30K test predictions and graph features

test_predictions = pd.read_csv(PREDICTIONS_TEST_PATH)
features_graph = pd.read_csv(FEATURES_GRAPH_PATH)

test_account_ids = test_predictions["account_id"].astype(str).unique()
features_graph["account_id"] = features_graph["account_id"].astype(str)

test_graph_df = features_graph[
    features_graph["account_id"].isin(test_account_ids)
].copy()

print("Test predictions:", len(test_predictions))
print("Unique test accounts:", len(test_account_ids))
print("Graph rows for test accounts:", len(test_graph_df))

assert len(test_graph_df) == len(test_account_ids), (
    "Number of test graph rows does not match unique test IDs."
)

Test predictions: 4509
Unique test accounts: 4509
Graph rows for test accounts: 4509


In [24]:
# Cell 6 — Load metrics and set threshold

with open(MODEL_METRICS_PATH, "r") as f:
    model_metrics = json.load(f)

print("30K Model metrics keys:")
print(model_metrics.keys())

# Current 30K Model B operating threshold from Day 6.
THRESHOLD_A = 0.010

print("Operating threshold:", THRESHOLD_A)

30K Model metrics keys:
dict_keys(['split', 'artifacts', 'model_A', 'model_B', 'baseline_test'])
Operating threshold: 0.01


In [25]:
assert len(test_predictions) == len(test_graph_df), (
    f"Prediction rows ({len(test_predictions)}) != "
    f"graph feature rows ({len(test_graph_df)})"
)

assert set(test_predictions["account_id"]) == set(
    test_graph_df["account_id"]
), "Prediction and graph feature account IDs do not match."

print("Prediction ↔ graph feature alignment: PASSED")

Prediction ↔ graph feature alignment: PASSED


In [26]:
# Cell 7 — Flag accounts using threshold


test_predictions["account_id"] = test_predictions["account_id"].astype(str)

# Use saved proba_A. If missing, compute from model_A.
if "proba_A" not in test_predictions.columns:
    X_test_all = test_graph_df[model_A_features].copy()
    test_predictions["proba_A"] = model_A.predict_proba(X_test_all)[:, 1]

proba_A = test_predictions["proba_A"].to_numpy()

# Rank based on probability
rank = (
    pd.Series(proba_A)
    .rank(ascending=False, method="first")
    .astype(int)
    .to_numpy()
)

test_predictions["rank"] = rank
test_predictions["investigation_flag"] = (
    test_predictions["proba_A"] >= THRESHOLD_A
)

flagged_df = test_predictions[test_predictions["investigation_flag"]].copy()
flagged_df = flagged_df.sort_values("proba_A", ascending=False)

n_flagged = len(flagged_df)

print("Test accounts:", len(test_predictions))
print("Flagged accounts:", n_flagged)
print("\nFirst 10 flagged accounts:")
print(flagged_df[["account_id", "rank", "proba_A"]].head(10).to_string(index=False))

Test accounts: 4509
Flagged accounts: 236

First 10 flagged accounts:
account_id  rank  proba_A
   A024295     1      1.0
   A024090     2      1.0
   A024050     3      1.0
   A024289     4      1.0
   A024156     5      1.0
   A024179     6      1.0
   A024154     7      1.0
   A024119     8      1.0
   A024180     9      1.0
   A024117    10      1.0


In [27]:
# Cell 8 — Prepare X_test for all test accounts

# Align full test set with Model B features
X_test_all = test_graph_df[model_A_features].copy()

assert list(X_test_all.columns) == model_A_features
assert X_test_all.select_dtypes(exclude="number").shape[1] == 0
assert np.isfinite(X_test_all.to_numpy()).all()

print("Full test graph rows:", len(test_graph_df))
print("X_test_all shape:", X_test_all.shape)

Full test graph rows: 4509
X_test_all shape: (4509, 40)


In [28]:
# Cell 9 — SHAP for all test accounts

explainer = shap.TreeExplainer(model_A)

shap_values_all = explainer.shap_values(X_test_all)

if isinstance(shap_values_all, list):
    shap_values_all = shap_values_all[1]

shap_values_all = np.asarray(shap_values_all)

print("SHAP values shape:", shap_values_all.shape)
print("X_test_all shape:", X_test_all.shape)

SHAP values shape: (4509, 40)
X_test_all shape: (4509, 40)


d:\CODIN PLAYGROUND\ML-AI\RingWatch\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [29]:
# Cell 10 — Build SHAP DataFrame for all test accounts

shap_df = pd.DataFrame(
    shap_values_all,
    columns=model_A_features,
    index=test_graph_df.index,
)

# Create lookup from test_predictions
rank_lookup = test_predictions.set_index("account_id")["rank"]
proba_lookup = test_predictions.set_index("account_id")["proba_A"]
flag_lookup = test_predictions.set_index("account_id")["investigation_flag"]

shap_df.insert(0, "account_id", test_graph_df["account_id"].values)
shap_df.insert(1, "rank", test_graph_df["account_id"].map(rank_lookup).values)
shap_df.insert(2, "proba", test_graph_df["account_id"].map(proba_lookup).values)
shap_df.insert(3, "investigation_flag", test_graph_df["account_id"].map(flag_lookup).values)

assert shap_df.shape[0] == len(X_test_all)
assert np.isfinite(shap_df[model_A_features].to_numpy()).all()

print("SHAP DataFrame rows:", len(shap_df))
print("SHAP DataFrame columns:", len(shap_df.columns))

shap_df.head()

SHAP DataFrame rows: 4509
SHAP DataFrame columns: 44


,account_id,rank,proba,investigation_flag,total_orders,total_amount,avg_order_value,distinct_devices,distinct_addresses,distinct_phones,...,account_creation_burst_score,accounts_per_device,shared_device_count,accounts_per_address,shared_address_count,accounts_per_phone,shared_phone_count,accounts_per_instrument,shared_instrument_count,shared_ip_prefix_count
23,A000023,1092,3.114439e-08,False,0.199875,0.090737,0.112369,0.073212,-0.454142,0.000609,...,0.017344,-0.215411,-0.052695,-0.370370,-0.545417,0.175117,0.019312,-0.133775,-0.028396,-0.207914
29,A000029,370,1.115540e-07,False,-0.658249,0.009271,0.106769,0.092828,0.150690,0.002796,...,-0.002670,-0.226242,-0.052308,-0.280723,0.190618,-0.059312,-0.022129,0.236110,0.023633,-0.166664
30,A000030,3974,2.155610e-08,False,0.180778,0.103385,0.098907,-0.324452,0.092356,0.000860,...,-0.006282,-0.286624,-0.102004,-0.348303,0.150058,-0.175820,-0.026179,-0.281031,-0.053281,-0.233497
31,A000031,2306,2.274596e-08,False,-0.462785,-0.190183,-0.205342,0.081577,0.112648,-0.003513,...,0.007772,0.238943,0.065342,0.467108,0.297858,0.097712,0.021538,0.129528,0.024178,-0.155405
43,A000043,4087,2.085334e-08,False,-0.215784,-0.040456,-0.093794,0.050958,0.072429,0.001954,...,-0.004561,0.204490,0.057782,-0.236674,0.080060,-0.130496,-0.023231,-0.085404,-0.016545,-0.107040


In [30]:
# Cell 11 — Save SHAP values

shap_df.to_csv(SHAP_VALUES_PATH, index=False)
print("Saved SHAP values to:")
print(SHAP_VALUES_PATH)

Saved SHAP values to:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\shap_values_test.csv


In [31]:
# Cell 12 — SHAP summary plot for flagged accounts

plt.figure(figsize=(12, 8))

shap.summary_plot(
    shap_values_all,
    X_test_all,
    feature_names=model_A_features,
    show=False,
)

plt.savefig(
    SHAP_SUMMARY_PATH,
    dpi=150,
    bbox_inches="tight",
)

plt.close()

print("SHAP summary plot saved:")
print(SHAP_SUMMARY_PATH)

SHAP summary plot saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\shap_summary.png


In [32]:
# Cell 13 — SHAP additivity check

expected_value = explainer.expected_value

if isinstance(expected_value, (list, np.ndarray)):
    expected_value = np.asarray(expected_value).reshape(-1)[-1]

shap_raw_score = shap_values_all.sum(axis=1) + expected_value

assert np.isfinite(shap_raw_score).all()

print("SHAP additivity check passed.")
print("Raw score range:")
print(shap_raw_score.min(), "→", shap_raw_score.max())

SHAP additivity check passed.
Raw score range:
-17.907934282352507 → 15.150012399096028


In [33]:
# Cell 14 — Top SHAP contributors helper

def get_top_shap_features(
    account_index,
    shap_values,
    feature_names,
    top_n=5,
):
    values = shap_values[account_index]

    ranking = (
        pd.DataFrame({
            "feature": feature_names,
            "shap_value": values,
        })
        .assign(abs_shap=lambda df: df["shap_value"].abs())
        .sort_values("abs_shap", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return ranking


# Example for first flagged account
example = get_top_shap_features(
    account_index=0,
    shap_values=shap_values_all,
    feature_names=model_A_features,
    top_n=5,
)

print("Top SHAP contributors for first flagged account:")
print(example.to_string(index=False))

Top SHAP contributors for first flagged account:
                  feature  shap_value  abs_shap
   total_delivered_orders    0.833096  0.833096
     shared_address_count   -0.545417  0.545417
discount_dependency_score   -0.458950  0.458950
       distinct_addresses   -0.454142  0.454142
     accounts_per_address   -0.370370  0.370370


In [34]:
# Cell 15 — Load 30K orders and disputes from versioned data

orders = pd.read_csv(
    PATHS["orders"],
    parse_dates=[
        "order_timestamp",
        "delivery_timestamp",
        "return_timestamp",
        "refund_timestamp",
    ],
)

disputes = pd.read_csv(
    PATHS["disputes"],
    parse_dates=["dispute_created_at"],
)

print("Orders:", len(orders))
print("Disputes:", len(disputes))
print("Dispute columns:")
print(disputes.columns.tolist())

Orders: 70027
Disputes: 570
Dispute columns:
['dispute_id', 'order_id', 'account_id', 'dispute_created_at', 'dispute_phase', 'dispute_reason_code', 'dispute_reason_category', 'respond_by', 'proof_of_service', 'explanation_letter', 'refund_confirmation', 'access_activity_log', 'refund_cancellation_policy', 'terms_and_conditions']


C:\Users\adity\AppData\Local\Temp\ipykernel_37280\2926735993.py:3: DtypeWarning: Columns (0: dispute_phase, 1: dispute_reason_code, 2: dispute_reason_category, 3: dispute_created_at) have mixed types. Specify dtype option on import or set low_memory=False.
  orders = pd.read_csv(


In [35]:
# Cell 16 — Filter orders/disputes to cutoff

# Force cutoff to pandas Timestamp
T = pd.to_datetime(PREDICTION_CUTOFF)

# Force timestamp columns to datetime as well
orders["order_timestamp"] = pd.to_datetime(
    orders["order_timestamp"],
    errors="coerce",
)

disputes["dispute_created_at"] = pd.to_datetime(
    disputes["dispute_created_at"],
    errors="coerce",
)

# Validate datetime types
print("T:", T)
print("T type:", type(T))
print("order_timestamp type:", orders["order_timestamp"].dtype)
print("dispute_created_at type:", disputes["dispute_created_at"].dtype)

# Filter to prediction cutoff
orders_pre = orders[
    orders["order_timestamp"] <= T
].copy()

disputes_pre = disputes[
    disputes["dispute_created_at"] <= T
].copy()

print("Orders before cutoff:", len(orders_pre))
print("Disputes before cutoff:", len(disputes_pre))

T: 2026-02-20 00:00:00
T type: <class 'pandas.Timestamp'>
order_timestamp type: datetime64[us]
dispute_created_at type: datetime64[us]
Orders before cutoff: 43720
Disputes before cutoff: 493


In [36]:
# Cell 17 — Evidence fields and helper

EVIDENCE_FIELDS = [
    "proof_of_service",
    "explanation_letter",
    "refund_confirmation",
    "access_activity_log",
    "refund_cancellation_policy",
    "terms_and_conditions",
]

missing_evidence = [col for col in EVIDENCE_FIELDS if col not in disputes_pre.columns]

if missing_evidence:
    raise ValueError(f"Missing evidence columns in disputes.csv: {missing_evidence}")

def normalize_evidence_value(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, np.integer)):
        return bool(value)
    if isinstance(value, float):
        return bool(value)
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"true", "yes", "available", "1"}:
            return True
        if v in {"false", "no", "missing", "0"}:
            return False
    return bool(value)

print("Evidence schema validated.")

Evidence schema validated.


In [37]:
# Cell 18 — Build evidence-gap table for flagged accounts

evidence_records = []

for account_id in flagged_df["account_id"]:
    account_disputes = disputes_pre[disputes_pre["account_id"] == account_id].copy()

    record = {"account_id": account_id}

    if account_disputes.empty:
        record["has_dispute_at_cutoff"] = False
        for field in EVIDENCE_FIELDS:
            record[field] = "NO_DISPUTE_YET"
        record["missing_evidence_count"] = None
    else:
        record["has_dispute_at_cutoff"] = True
        account_disputes = account_disputes.sort_values("dispute_created_at")
        latest = account_disputes.iloc[-1]

        missing_count = 0
        for field in EVIDENCE_FIELDS:
            available = normalize_evidence_value(latest[field])
            record[field] = available
            if not available:
                missing_count += 1
        record["missing_evidence_count"] = missing_count

    evidence_records.append(record)

evidence_df = pd.DataFrame(evidence_records)

print("Evidence rows:", len(evidence_df))

Evidence rows: 236


In [38]:
# Cell 19 — Validate and save evidence gaps

assert len(evidence_df) == len(flagged_df)
assert evidence_df["account_id"].is_unique

evidence_df.to_csv(EVIDENCE_GAP_PATH, index=False)

print("Evidence gaps saved to:")
print(EVIDENCE_GAP_PATH)

Evidence gaps saved to:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\evidence_gap_test.csv


In [39]:
# Cell 20 — Load 30K account graph edges

GRAPH_EDGES_PATH = Path(PATHS["orders"]).parent / "processed" / "account_graph_edges.csv"

edges = pd.read_csv(GRAPH_EDGES_PATH)

print("Graph edges:", len(edges))
print(edges.head())

Graph edges: 139417
  account_id_1 account_id_2       edge_type  weight
0      A000000      A001697    shares_phone     1.0
1      A000000      A002763  shares_address     0.7
2      A000000      A002989  shares_address     0.7
3      A000000      A004877  shares_address     0.7
4      A000000      A006207   shares_device     1.0


In [40]:
# Cell 21 — Graph evidence helper

def get_graph_evidence(account_id, edges):
    account_edges = edges[
        (edges["account_id_1"] == account_id) |
        (edges["account_id_2"] == account_id)
    ].copy()

    if account_edges.empty:
        return {
            "account_id": account_id,
            "total_graph_links": 0,
            "strongest_edge_type": None,
            "strongest_edge_weight": None,
            "number_of_device_links": 0,
            "number_of_ip_links": 0,
            "number_of_coupon_links": 0,
            "linked_accounts": "",
        }

    linked_accounts = []

    for _, edge in account_edges.iterrows():
        if edge["account_id_1"] == account_id:
            linked = edge["account_id_2"]
        else:
            linked = edge["account_id_1"]

        linked_accounts.append(f"{edge['edge_type']} -> {linked}")

    strongest = account_edges.loc[account_edges["weight"].idxmax()]

    return {
        "account_id": account_id,
        "total_graph_links": len(account_edges),
        "strongest_edge_type": strongest["edge_type"],
        "strongest_edge_weight": strongest["weight"],
        "number_of_device_links": int((account_edges["edge_type"] == "shares_device").sum()),
        "number_of_ip_links": int((account_edges["edge_type"] == "shares_ip_prefix").sum()),
        "number_of_coupon_links": int((account_edges["edge_type"] == "shares_coupon").sum()),
        "linked_accounts": " | ".join(linked_accounts),
    }

In [41]:
# Cell 22 — Generate graph evidence for flagged accounts

graph_records = []

for account_id in flagged_df["account_id"]:
    graph_records.append(get_graph_evidence(account_id, edges))

graph_evidence_df = pd.DataFrame(graph_records)

print("Graph evidence rows:", len(graph_evidence_df))

graph_evidence_df.to_csv(GRAPH_EVIDENCE_PATH, index=False)

print("Graph evidence saved:")
print(GRAPH_EVIDENCE_PATH)

Graph evidence rows: 236
Graph evidence saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\graph_evidence_test.csv


In [42]:
# Cell 23 — Save individual graph PNGs for top 5 flagged accounts

TOP_N_GRAPHS = 5

graph_png_dir = EXPLAINABILITY_DIR / "graphs"
graph_png_dir.mkdir(parents=True, exist_ok=True)

for _, row in flagged_df.head(TOP_N_GRAPHS).iterrows():
    acc = row["account_id"]
    acc_edges = edges[
        (edges["account_id_1"] == acc) |
        (edges["account_id_2"] == acc)
    ]

    G = nx.Graph()
    G.add_node(acc, focus=True)

    for _, edge in acc_edges.iterrows():
        other = edge["account_id_2"] if edge["account_id_1"] == acc else edge["account_id_1"]
        G.add_node(other, focus=False)
        G.add_edge(acc, other, edge_type=edge["edge_type"])

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)

    node_colors = ["black" if G.nodes[n].get("focus") else "gray" for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=300)

    nx.draw_networkx_edges(G, pos, edge_color="gray", width=1.0)

    labels = {n: n for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)

    edge_labels = {
        (u, v): d["edge_type"].replace("shares_", "")
        for u, v, d in G.edges(data=True)
    }
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6)

    plt.title(f"Graph for {acc}")
    plt.axis("off")

    png_path = graph_png_dir / f"{acc}_graph.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"Saved graph: {png_path}")

Saved graph: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\graphs\A024295_graph.png
Saved graph: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\graphs\A024090_graph.png
Saved graph: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\graphs\A024050_graph.png
Saved graph: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\graphs\A024289_graph.png
Saved graph: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\graphs\A024156_graph.png


In [43]:
# Cell 24 — Build case-report data

case_df = flagged_df[["account_id", "rank", "proba_B"]].rename(columns={"proba_B": "proba"}).copy()

case_df = case_df.merge(evidence_df, on="account_id", how="left", validate="one_to_one")
case_df = case_df.merge(graph_evidence_df, on="account_id", how="left", validate="one_to_one")

case_df.head()

,account_id,rank,proba,has_dispute_at_cutoff,proof_of_service,explanation_letter,refund_confirmation,access_activity_log,refund_cancellation_policy,terms_and_conditions,missing_evidence_count,total_graph_links,strongest_edge_type,strongest_edge_weight,number_of_device_links,number_of_ip_links,number_of_coupon_links,linked_accounts
0,A024295,1,1.0,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NaN,7,shares_device,1.0,7,0,0,shares_device -> A024288 | shares_device -> A0...
1,A024090,2,1.0,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NaN,7,shares_device,1.0,7,0,0,shares_device -> A024088 | shares_device -> A0...
2,A024050,3,1.0,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NaN,7,shares_device,1.0,7,0,0,shares_device -> A024048 | shares_device -> A0...
3,A024289,4,1.0,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NaN,7,shares_device,1.0,7,0,0,shares_device -> A024288 | shares_device -> A0...
4,A024156,5,1.0,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NaN,7,shares_device,1.0,7,0,0,shares_device -> A024152 | shares_device -> A0...


In [44]:
# Cell 25 — Create SHAP summaries for flagged accounts

shap_summaries = {}

# Build a direct account_id -> SHAP row-position mapping.
# shap_values_all corresponds exactly to test_graph_df.
shap_position_lookup = {
    account_id: idx
    for idx, account_id in enumerate(test_graph_df["account_id"])
}

for account_id in flagged_df["account_id"]:
    if account_id not in shap_position_lookup:
        raise KeyError(
            f"Flagged account {account_id} not found in test_graph_df."
        )

    pos_idx = shap_position_lookup[account_id]

    top_features = get_top_shap_features(
        account_index=pos_idx,
        shap_values=shap_values_all,
        feature_names=model_A_features,
        top_n=5,
    )

    shap_summaries[account_id] = top_features

print("SHAP summaries created:", len(shap_summaries))

SHAP summaries created: 236


In [45]:
# Cell 26 — Rank-based risk tier and action

def get_risk_tier(rank, community_size):
    if rank <= 5:
        if community_size >= 4:
            return "CRITICAL"
        return "HIGH"

    if rank <= 20:
        return "HIGH"

    if rank <= 60:
        return "MEDIUM"

    return "LOW"


def recommend_action(risk_tier):
    if risk_tier == "CRITICAL":
        return "CRITICAL: recommend human review + soft-hold refunds"

    if risk_tier == "HIGH":
        return "HIGH: recommend human review"

    if risk_tier == "MEDIUM":
        return "MEDIUM: recommend step-up verification on refund"

    return "LOW: monitor — no immediate refund action"

In [46]:
# Cell 27 — Generate bounded actions

actions_df = case_df[
    ["account_id", "rank", "proba"]
].copy()

community_lookup = (
    test_graph_df
    .set_index("account_id")["community_size"]
    .to_dict()
)

actions_df["community_size"] = (
    actions_df["account_id"]
    .map(community_lookup)
    .fillna(0)
)


actions_df["risk_tier"] = actions_df.apply(
    lambda row: get_risk_tier(
        row["rank"],
        row["community_size"],
    ),
    axis=1,
)


actions_df["recommended_action"] = (
    actions_df["risk_tier"]
    .apply(recommend_action)
)


actions_df = actions_df[
    [
        "account_id",
        "rank",
        "proba",
        "risk_tier",
        "recommended_action",
    ]
].sort_values("rank")


print("Actions DataFrame:")
print(actions_df.to_string(index=False))

print("\nColumns:")
print(actions_df.columns.tolist())

print("\nFlagged accounts:", n_flagged)
print("Action rows:", len(actions_df))

Actions DataFrame:
account_id  rank    proba risk_tier                                   recommended_action
   A024295     1 1.000000  CRITICAL CRITICAL: recommend human review + soft-hold refunds
   A024090     2 1.000000  CRITICAL CRITICAL: recommend human review + soft-hold refunds
   A024050     3 1.000000  CRITICAL CRITICAL: recommend human review + soft-hold refunds
   A024289     4 1.000000  CRITICAL CRITICAL: recommend human review + soft-hold refunds
   A024156     5 1.000000  CRITICAL CRITICAL: recommend human review + soft-hold refunds
   A024179     6 1.000000      HIGH                         HIGH: recommend human review
   A024154     7 1.000000      HIGH                         HIGH: recommend human review
   A024119     8 1.000000      HIGH                         HIGH: recommend human review
   A024180     9 1.000000      HIGH                         HIGH: recommend human review
   A024117    10 1.000000      HIGH                         HIGH: recommend human review
  

In [47]:
# Cell 28 — Validate and save bounded actions

allowed_tiers = {
    "CRITICAL",
    "HIGH",
    "MEDIUM",
    "LOW",
}

assert len(actions_df) == n_flagged

assert actions_df["account_id"].is_unique

assert set(
    actions_df["risk_tier"]
).issubset(allowed_tiers)

assert "recommended_action" in actions_df.columns

assert actions_df["recommended_action"].notna().all()


forbidden = [
    "block account",
    "ban account",
    "permanent ban",
    "automatically reject",
]

action_text = (
    actions_df["recommended_action"]
    .str.lower()
)

for term in forbidden:
    assert not action_text.str.contains(
        term,
        regex=False,
    ).any(), f"Forbidden action: {term}"


actions_df.to_csv(
    BOUNDED_ACTIONS_PATH,
    index=False,
)


print("Bounded actions saved:")
print(BOUNDED_ACTIONS_PATH)

print("Rows:", len(actions_df))

Bounded actions saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\bounded_actions_test.csv
Rows: 236


In [48]:
# Cell 29 — Merge actions into case_df and generate case reports

case_df = case_df.merge(
    actions_df[
        ["account_id", "risk_tier", "recommended_action"]
    ],
    on="account_id",
    how="left",
)


def build_case_report(row, shap_summary):
    lines = [
        f"Account ID: {row['account_id']}",
        f"Risk score (Model A): {row['proba']:.6f}",
        f"Investigation rank: {int(row['rank'])} / {n_flagged} flagged accounts",
        "",
        "Observed facts:",
    ]

    account_row = test_graph_df[
        test_graph_df["account_id"] == row["account_id"]
    ].iloc[0]

    behavioral_fields = [
        "total_orders",
        "return_rate",
        "refund_rate",
        "dispute_rate",
        "shared_device_count",
        "shared_ip_prefix_count",
        "community_size",
    ]

    for field in behavioral_fields:
        value = account_row.get(field, "N/A")
        lines.append(f"  {field}: {value}")

    lines.append("")
    lines.append("Top model contributors:")

    for _, shap_row in shap_summary.iterrows():
        direction = (
            "increased"
            if shap_row["shap_value"] > 0
            else "decreased"
        )

        lines.append(
            f"  {shap_row['feature']}: "
            f"{shap_row['shap_value']:.6f} "
            f"({direction} model risk)"
        )

    lines.append("")
    lines.append("Graph evidence:")

    if row["total_graph_links"] == 0:
        lines.append("  No graph relationships observed.")
    else:
        for rel in str(row["linked_accounts"]).split(" | "):
            lines.append(f"  {rel}")

    lines.append("")
    lines.append("Evidence status:")

    if row["has_dispute_at_cutoff"]:
        for field in EVIDENCE_FIELDS:
            status = (
                "AVAILABLE"
                if row[field] is True
                else "MISSING"
            )

            lines.append(
                f"  {field}: {status}"
            )

        lines.append(
            f"  Missing evidence count: "
            f"{int(row['missing_evidence_count'])}"
        )

    else:
        lines.append(
            "  No dispute observed at prediction cutoff."
        )

    lines.append("")
    lines.append("Recommended action:")
    lines.append(
        f"  {row['recommended_action']}"
    )

    return "\n".join(lines)


case_reports = []

for _, row in case_df.iterrows():

    account_id = row["account_id"]

    if account_id not in shap_summaries:
        raise KeyError(
            f"Missing SHAP summary for flagged account: "
            f"{account_id}"
        )

    report = build_case_report(
        row,
        shap_summaries[account_id],
    )

    case_reports.append(
        {
            "account_id": account_id,
            "rank": int(row["rank"]),
            "proba": float(row["proba"]),
            "case_report_text": report,
        }
    )


case_reports_df = pd.DataFrame(case_reports)


assert len(case_reports_df) == n_flagged
assert case_reports_df["account_id"].is_unique


case_reports_df.to_csv(
    CASE_REPORTS_PATH,
    index=False,
)


print("Case reports saved:")
print(CASE_REPORTS_PATH)
print("Case reports generated:", len(case_reports_df))

Case reports saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\case_reports_test.csv
Case reports generated: 236


In [49]:
# Cell 30 — Create investigation audit log

audit_timestamp = pd.Timestamp.now()

audit_df = actions_df[["account_id", "proba", "rank", "risk_tier", "recommended_action"]].copy()
audit_df.insert(0, "timestamp", audit_timestamp)
audit_df.insert(2, "model_version", "LightGBM_Model_A")
audit_df["top_k_flag"] = True
audit_df["action_recommended"] = audit_df["recommended_action"]
audit_df["case_report_generated"] = audit_df["account_id"].isin(case_reports_df["account_id"])

audit_df = audit_df[
    [
        "timestamp", "account_id", "model_version", "proba", "rank",
        "risk_tier", "top_k_flag", "action_recommended", "case_report_generated",
    ]
]

audit_df.to_csv(AUDIT_LOG_PATH, index=False)

print("Audit log saved:")
print(AUDIT_LOG_PATH)

Audit log saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v3_scaled_30k\processed\explainability\investigation_audit_log.csv


In [50]:
# Cell 31 — Final validation using n_flagged

expected_files = [
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
]

for path in expected_files:
    assert Path(path).exists(), f"Missing output: {path}"

assert len(flagged_df) == n_flagged
assert len(evidence_df) == n_flagged
assert len(graph_evidence_df) == n_flagged
assert len(case_reports_df) == n_flagged
assert len(actions_df) == n_flagged
assert len(audit_df) == n_flagged

print("=" * 55)
print("RINGWATCH DAY 8–9 VALIDATION (30K)")
print("=" * 55)
print(f"Investigation queue: {n_flagged}")
print(f"SHAP explanations:   {len(shap_df)}")
print(f"Evidence reports:    {len(evidence_df)}")
print(f"Graph evidence:      {len(graph_evidence_df)}")
print(f"Case reports:        {len(case_reports_df)}")
print(f"Actions:             {len(actions_df)}")
print(f"Audit records:       {len(audit_df)}")
print("DAY 8–9 PASSED")

RINGWATCH DAY 8–9 VALIDATION (30K)
Investigation queue: 236
SHAP explanations:   4509
Evidence reports:    236
Graph evidence:      236
Case reports:        236
Actions:             236
Audit records:       236
DAY 8–9 PASSED


In [51]:
# Cell 31 — Final validation using n_flagged

expected_files = [
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
]

for path in expected_files:
    assert Path(path).exists(), f"Missing output: {path}"

assert len(flagged_df) == n_flagged
assert len(evidence_df) == n_flagged
assert len(graph_evidence_df) == n_flagged
assert len(case_reports_df) == n_flagged
assert len(actions_df) == n_flagged
assert len(audit_df) == n_flagged

print("=" * 55)
print("RINGWATCH DAY 8–9 VALIDATION (30K)")
print("=" * 55)
print(f"Investigation queue: {n_flagged}")
print(f"SHAP explanations:   {len(shap_df)}")
print(f"Evidence reports:    {len(evidence_df)}")
print(f"Graph evidence:      {len(graph_evidence_df)}")
print(f"Case reports:        {len(case_reports_df)}")
print(f"Actions:             {len(actions_df)}")
print(f"Audit records:       {len(audit_df)}")
print("DAY 8–9 PASSED")

RINGWATCH DAY 8–9 VALIDATION (30K)
Investigation queue: 236
SHAP explanations:   4509
Evidence reports:    236
Graph evidence:      236
Case reports:        236
Actions:             236
Audit records:       236
DAY 8–9 PASSED


In [52]:
# Cell 32 — Show final investigation queue
display(
    case_df[
        [
            "account_id", "rank", "proba", "risk_tier",
            "recommended_action", "total_graph_links", "missing_evidence_count",
        ]
    ]
    .sort_values("rank")
    .head(20)
)

,account_id,rank,proba,risk_tier,recommended_action,total_graph_links,missing_evidence_count
0,A024295,1,1.0,CRITICAL,CRITICAL: recommend human review + soft-hold r...,7,NaN
1,A024090,2,1.0,CRITICAL,CRITICAL: recommend human review + soft-hold r...,7,NaN
2,A024050,3,1.0,CRITICAL,CRITICAL: recommend human review + soft-hold r...,7,NaN
3,A024289,4,1.0,CRITICAL,CRITICAL: recommend human review + soft-hold r...,7,NaN
4,A024156,5,1.0,CRITICAL,CRITICAL: recommend human review + soft-hold r...,7,NaN
5,A024179,6,1.0,HIGH,HIGH: recommend human review,7,NaN
6,A024154,7,1.0,HIGH,HIGH: recommend human review,7,NaN
7,A024119,8,1.0,HIGH,HIGH: recommend human review,7,NaN
8,A024180,9,1.0,HIGH,HIGH: recommend human review,7,NaN
9,A024117,10,1.0,HIGH,HIGH: recommend human review,7,NaN
